# 🌐 Projeto Web Scraping Wikipédia & Data App
**Disciplina / Tarefa Acadêmica - UFRN**  
**Objetivo:** Coleta de dados via web scraping (Requests+BS4 vs Scrapy+Crochet), processamento de linguagem natural (NLTK stopwords), geração de nuvem de palavras e contagem de termos em corpus consolidado.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)

## 1. Instalação e Configuração das Dependências

In [ ]:
!pip install requests beautifulsoup4 scrapy crochet wordcloud matplotlib nltk pandas plotly streamlit

In [ ]:
import time
import urllib.parse
import re
import os
from collections import Counter
import requests
from bs4 import BeautifulSoup
import matplotlib.pyplot as plt
from wordcloud import WordCloud
import nltk
import pandas as pd

# Baixar stopwords em português
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
from nltk.corpus import stopwords
STOPWORDS_PT = set(stopwords.words('portuguese'))
STOPWORDS_PT.update({'artigo', 'artigos', 'links', 'externos', 'referências', 'ver', 'também', 'página', 'wikipédia', 'conteúdo', 'sobre'})

print("✓ Dependências e stopwords configuradas com sucesso!")

## 2. Utilitários de Limpeza de Texto e Nuvem de Palavras

In [ ]:
def clean_text_portuguese(raw_text: str, remove_stopwords: bool = True) -> tuple[str, list[str]]:
    if not raw_text:
        return "", []
    # Remove referências [1], [nota 1]
    text = re.sub(r'\[.*?\]', ' ', raw_text).lower()
    # Remove caracteres especiais e pontuações
    text = re.sub(r'[^a-záàâãéèêíïóôõöúçñ\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    tokens = text.split()
    if remove_stopwords:
        tokens = [w for w in tokens if w not in STOPWORDS_PT and len(w) > 2]
    return " ".join(tokens), tokens

def plot_wordcloud(text: str, title: str = "Nuvem de Palavras", colormap: str = "viridis"):
    wc = WordCloud(width=900, height=450, background_color='#111827', colormap=colormap, stopwords=STOPWORDS_PT).generate(text)
    plt.figure(figsize=(10, 5), facecolor='#111827')
    plt.imshow(wc, interpolation='bilinear')
    plt.axis('off')
    plt.title(title, fontsize=16, color='white', pad=12)
    plt.show()

## 3. Parte 1 — Solução com Requests + BeautifulSoup
- Medição do tempo de execução com `time.perf_counter()`
- Extração das tags `<p>` da Wikipédia
- Geração da Nuvem de Palavras

In [ ]:
def scrape_bs4(term: str) -> dict:
    formatted_term = term.strip().replace(' ', '_')
    encoded = urllib.parse.quote(formatted_term, safe=':/_')
    url = f"https://pt.wikipedia.org/wiki/{encoded}"
    headers = {"User-Agent": "WikiDataApp-UFRN/1.0 (academic research; contato@ufrn.br)"}
    
    t_start = time.perf_counter()
    resp = requests.get(url, headers=headers, timeout=15)
    resp.raise_for_status()
    
    soup = BeautifulSoup(resp.content, 'html.parser')
    content = soup.find('div', {'id': 'bodyContent'}) or soup
    paragraphs = [p.get_text().strip() for p in content.find_all('p') if len(p.get_text().strip()) > 5]
    t_elapsed = time.perf_counter() - t_start
    
    full_text = "\n\n".join(paragraphs)
    return {
        "term": term,
        "url": resp.url,
        "text": full_text,
        "paragraphs": paragraphs,
        "time": round(t_elapsed, 4),
        "words": len(full_text.split())
    }

# Teste com 'Ciência de dados'
res_bs4 = scrape_bs4("Ciência de dados")
print(f"⏱️ Tempo de execução (BS4): {res_bs4['time']}s | Parágrafos: {len(res_bs4['paragraphs'])} | Palavras: {res_bs4['words']}")

cleaned_bs4, tokens_bs4 = clean_text_portuguese(res_bs4['text'])
plot_wordcloud(cleaned_bs4, title=f"WordCloud BS4: {res_bs4['term']}")

## 4. Parte 2 — Solução com Scrapy + Crochet
- Inicialização do `crochet.setup()` para desacoplar o reactor Twisted no Colab
- Spider Scrapy com anotação `@wait_for`
- Medição e comparação de tempo de execução

In [ ]:
import crochet
import scrapy
from scrapy.crawler import CrawlerRunner

# Inicializar o Crochet
crochet.setup()

runner = CrawlerRunner({
    'USER_AGENT': 'WikiDataApp-UFRN/1.0 (academic research; contato@ufrn.br)',
    'LOG_LEVEL': 'ERROR',
    'TWISTED_REACTOR': 'twisted.internet.epollreactor.EPollReactor',
    'ROBOTSTXT_OBEY': False
})

class ColabWikiSpider(scrapy.Spider):
    name = 'colab_wiki_spider'
    def __init__(self, target_url, items, *args, **kwargs):
        super(ColabWikiSpider, self).__init__(*args, **kwargs)
        self.start_urls = [target_url]
        self.items = items
        
    def parse(self, response):
        paragraphs = []
        for p in response.xpath("//div[@id='bodyContent']//p | //p"):
            txt = p.xpath("string(.)").get()
            if txt and len(txt.strip()) > 5:
                paragraphs.append(txt.strip())
        item = {'url': response.url, 'paragraphs': paragraphs}
        self.items.append(item)
        yield item

@crochet.wait_for(timeout=30.0)
def _run_scrapy(target_url: str):
    items = []
    deferred = runner.crawl(ColabWikiSpider, target_url=target_url, items=items)
    deferred.addBoth(lambda _: items)
    return deferred

def scrape_scrapy(term: str) -> dict:
    formatted_term = term.strip().replace(' ', '_')
    encoded = urllib.parse.quote(formatted_term, safe=':/_')
    url = f"https://pt.wikipedia.org/wiki/{encoded}"
    
    t_start = time.perf_counter()
    items = _run_scrapy(url)
    t_elapsed = time.perf_counter() - t_start
    
    paragraphs = items[0]['paragraphs'] if items else []
    full_text = "\n\n".join(paragraphs)
    return {
        "term": term,
        "url": url,
        "text": full_text,
        "paragraphs": paragraphs,
        "time": round(t_elapsed, 4),
        "words": len(full_text.split())
    }

# Teste com Scrapy
res_scrapy = scrape_scrapy("Ciência de dados")
print(f"⚡ Tempo de execução (Scrapy): {res_scrapy['time']}s | Parágrafos: {len(res_scrapy['paragraphs'])} | Palavras: {res_scrapy['words']}")

cleaned_scrapy, tokens_scrapy = clean_text_portuguese(res_scrapy['text'])
plot_wordcloud(cleaned_scrapy, title=f"WordCloud Scrapy: {res_scrapy['term']}", colormap="plasma")

## 5. Parte 3 — Raspagem Múltipla (5 Termos), Limpeza e Contagem de Palavras

In [ ]:
cinco_termos = [
    "Universidade Federal do Rio Grande do Norte",
    "Ciência de Dados",
    "Aprendizado de Máquina",
    "Engenharia de Software",
    "Armazém de Dados"
]

print("Iniciando raspagem dos 5 termos...")
corpus_bruto = []
for termo in cinco_termos:
    print(f" -> Coletando: {termo}...")
    r = scrape_bs4(termo)
    corpus_bruto.append(r["text"])

texto_unificado_bruto = "\n\n".join(corpus_bruto)
texto_limpo, tokens_limpos = clean_text_portuguese(texto_unificado_bruto, remove_stopwords=True)

print("\n--- Estatísticas do Corpus ---")
print(f"Total de palavras brutas: {len(texto_unificado_bruto.split()):,}")
print(f"Total de palavras limpas (sem stopwords): {len(tokens_limpos):,}")

# Nuvem de palavras dos 5 artigos
plot_wordcloud(texto_limpo, title="Nuvem de Palavras do Corpus Consolidado (5 Termos)", colormap="magma")

### Contagem de Palavra Específica no Corpus das 5 Páginas

In [ ]:
palavra_busca = "dados"
palavra_normalizada = palavra_busca.strip().lower()
ocorrencias = tokens_limpos.count(palavra_normalizada)
frequencia_relativa = (ocorrencias / len(tokens_limpos)) * 100

print(f"=======================================================")
print(f"A palavra '{palavra_busca}' apareceu {ocorrencias} vezes no corpus das 5 páginas.")
print(f"Frequência relativa: {frequencia_relativa:.2f}% de todas as palavras filtradas.")
print(f"=======================================================")

# Top 10 palavras mais frequentes
top_10 = Counter(tokens_limpos).most_common(10)
df_top = pd.DataFrame(top_10, columns=["Palavra", "Frequência"])
print("\nTop 10 palavras mais frequentes:")
display(df_top)